In [3]:
import json
import os
from pathlib import Path
from PIL import Image

In [4]:
DATA_DIR = Path("../../data/processed")

TRAIN_IMAGES = DATA_DIR / "train/images"
TRAIN_ANN = DATA_DIR / "train/annotations"

VAL_IMAGES = DATA_DIR / "validation/images"
VAL_ANN = DATA_DIR / "validation/annotations"


In [2]:
def find_corrupted_images(folder):
    
    corrupted = []
    
    for img_path in folder.glob("*"):
        
        try:
            img = Image.open(img_path)
            img.verify()
            
        except:
            corrupted.append(img_path)
    
    print("Total images:", len(list(folder.glob("*"))))
    print("Corrupted images:", len(corrupted))
    
    return corrupted


train_bad = find_corrupted_images(TRAIN_IMAGES)
val_bad = find_corrupted_images(VAL_IMAGES)


NameError: name 'TRAIN_IMAGES' is not defined

In [4]:
def find_corrupted_json(folder):
    
    corrupted = []
    
    for file in folder.glob("*.json"):
        
        try:
            with open(file) as f:
                json.load(f)
                
        except:
            corrupted.append(file)
    
    print("Total json:", len(list(folder.glob('*.json'))))
    print("Corrupted json:", len(corrupted))
    
    return corrupted


train_bad_json = find_corrupted_json(TRAIN_ANN)
val_bad_json = find_corrupted_json(VAL_ANN)


Total json: 191961
Corrupted json: 0
Total json: 32153
Corrupted json: 0


In [5]:
def check_pairs(images_folder, ann_folder):
    
    images = {p.stem for p in images_folder.glob("*")}
    anns = {p.stem for p in ann_folder.glob("*.json")}
    
    missing_ann = images - anns
    missing_img = anns - images
    
    print("Images:", len(images))
    print("Annotations:", len(anns))
    print("Images without annotation:", len(missing_ann))
    print("Annotations without image:", len(missing_img))
    
    return missing_ann, missing_img


train_missing = check_pairs(TRAIN_IMAGES, TRAIN_ANN)
val_missing = check_pairs(VAL_IMAGES, VAL_ANN)


Images: 191961
Annotations: 191961
Images without annotation: 0
Annotations without image: 0
Images: 32153
Annotations: 32153
Images without annotation: 0
Annotations without image: 0


In [5]:
categories = {}

for f in list(TRAIN_ANN.glob("*.json"))[:500]:
    data = json.load(open(f))
    for key, value in data.items():
        if key.startswith("item"):
            cid = value.get("category_id")
            cname = value.get("category_name")
            if cid and cname:
                categories[cid] = cname

for cid in sorted(categories):
    print(cid, "-", categories[cid])

1 - short sleeve top
2 - long sleeve top
3 - short sleeve outwear
4 - long sleeve outwear
5 - vest
6 - sling
7 - shorts
8 - trousers
9 - skirt
10 - short sleeve dress
11 - long sleeve dress
12 - vest dress
13 - sling dress


In [6]:
def check_rgb(folder, n=1000):
    bad = 0
    for img_path in list(folder.glob("*"))[:n]:
        img = Image.open(img_path)
        if img.mode != "RGB":
            bad += 1
    print(f"Out of {n} images, {bad} are not RGB.")

check_rgb(TRAIN_IMAGES)
check_rgb(VAL_IMAGES)


Out of 1000 images, 0 are not RGB.
Out of 1000 images, 0 are not RGB.
